# SQL Business Analysis — UCI Online Retail 

## Overview

This notebook uses SQL (via DuckDB) to answer four business questions about customer behaviour in the UCI Online Retail dataset.

It builds directly on the RFM + KMeans segmentation in `01_customer_analytics.ipynb`, importing cluster labels to cross-analyse retention and purchase behaviour at the segment level.

**Four business questions, in order of analytical dependency:**

| # | Question | Why it matters |
|---|----------|---------------|
| 1 | What share of customers ever return? | Establishes the baseline retention problem |
| 2 | When do customers return — and when do they stop? | Identifies the intervention window |
| 3 | Where in the purchase funnel does the business lose customers? | Quantifies the steepest drop-off point |
| 4 | Do different customer segments behave differently — and what does that imply for tactics? | Translates the population-level findings into segment-specific actions |

Each case builds on the previous one. By Case 4, the analysis has enough context to recommend specific, data-grounded tactics for each segment — not generic best practices.


## Dataset

The dataset contains transactional records from a UK-based online retailer, covering December 2010 – December 2011.

| Field | Description |
|---|---|
| `CustomerID` | Unique customer identifier |
| `InvoiceNo` | Transaction ID (prefix `C` = cancellation) |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units per line item |
| `UnitPrice` | Price per unit (GBP) |
| `InvoiceDate` | Timestamp of transaction |
| `Country` | Customer country |

Each row represents a single product line within a transaction.

## Setup

Connect to DuckDB and load the dataset. Rows with missing `CustomerID`, cancelled orders (InvoiceNo starting with `C`), or non-positive Quantity / UnitPrice are removed before analysis.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect()

In [3]:
df = pd.read_csv("online_retail.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'online_retail.csv'

In [9]:
df.shape

(541909, 8)

In [5]:
df_clean = df.copy()

# remove missing customer
df_clean = df_clean.dropna(subset=['CustomerID'])

# remove cancelled orders
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# remove invalid values
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

df_clean.shape


(397884, 8)

In [6]:
con.register("retail", df_clean)

---

## Case 1: Overall Repeat Purchase Rate

### Business question
What share of customers ever make a second purchase?

A high repeat rate suggests the business builds loyalty. A low rate suggests it runs on acquisition — expensive and fragile. This is the starting point before asking *when* or *which customers* return.


In [18]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS order_count
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE order_count > 1) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders
"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


In [20]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

customer_orders AS (
    SELECT 
        r.CustomerID,
        COUNT(DISTINCT DATE(r.InvoiceDate)) AS active_days,
        SUM(CASE WHEN DATE(r.InvoiceDate) > f.first_date THEN 1 ELSE 0 END) AS repeat_activity
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
    GROUP BY r.CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_activity > 0) * 1.0 
        / COUNT(*) AS repeat_purchase_rate
FROM customer_orders


"""

con.execute(query).fetchdf()

,repeat_purchase_rate
0,0.643154


**64.3% of customers make at least one repeat purchase** across the full observation window (Dec 2010 – Dec 2011).

This looks healthy — but it is a lifetime metric. A customer who returned once in 12 months counts the same as one who returned every month. The next question is *when* they return, because timing determines whether intervention is even possible.


---

## Case 2: Return Timing — When Do Customers Come Back?

### Business question
Of the customers who do return, when do they return? Is there a window where the business can realistically intervene?

If most customers return within 30 days naturally, a coupon adds little value. If the return window extends to 60–90 days, there is a meaningful intervention opportunity — but only if the nudge arrives before the customer has fully lapsed.


### Time-windowed repeat rate (30 / 60 / 90 days post first purchase)

In [21]:
query = """
WITH first_purchase AS (
    SELECT 
        CustomerID,
        MIN(DATE(InvoiceDate)) AS first_date
    FROM retail
    GROUP BY CustomerID
),

purchases AS (
    SELECT 
        r.CustomerID,
        DATE(r.InvoiceDate) AS purchase_date,
        f.first_date
    FROM retail r
    JOIN first_purchase f
        ON r.CustomerID = f.CustomerID
),

flags AS (
    SELECT 
        CustomerID,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 30 DAY 
            THEN 1 ELSE 0 END) AS repeat_30d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 60 DAY 
            THEN 1 ELSE 0 END) AS repeat_60d,

        MAX(CASE 
            WHEN purchase_date > first_date 
             AND purchase_date <= first_date + INTERVAL 90 DAY 
            THEN 1 ELSE 0 END) AS repeat_90d

    FROM purchases
    GROUP BY CustomerID
)

SELECT 
    COUNT(*) FILTER (WHERE repeat_30d = 1) * 1.0 / COUNT(*) AS repeat_30d_rate,
    COUNT(*) FILTER (WHERE repeat_60d = 1) * 1.0 / COUNT(*) AS repeat_60d_rate,
    COUNT(*) FILTER (WHERE repeat_90d = 1) * 1.0 / COUNT(*) AS repeat_90d_rate
FROM flags
"""
con.execute(query).fetchdf()

,repeat_30d_rate,repeat_60d_rate,repeat_90d_rate
0,0.187183,0.335639,0.422084


| Window | Cumulative return rate | Incremental gain |
|--------|----------------------|-----------------|
| 0 – 30 days | 18.7% | baseline |
| 0 – 60 days | 33.6% | **+14.9 pp** ← largest single jump |
| 0 – 90 days | 42.2% | +8.6 pp |

**The 31–60 day window captures nearly twice the incremental return rate of the 61–90 day window.** This is not random — it aligns with the inter-purchase gap distribution below.


In [3]:
# csv for tableau
retention_windows = pd.DataFrame({
    'Window': ['0-30d', '0-60d', '0-90d'],
    'Rate': [0.187, 0.336, 0.422],
    'Incremental': [0.187, 0.149, 0.086]
})
retention_windows.to_csv('retention_windows.csv', index=False)

### Inter-purchase gap distribution (all repeat buyers)

In [12]:
query = """
WITH order_dates AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        DATE(InvoiceDate) AS purchase_date,
        LAG(DATE(InvoiceDate)) OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS prev_date
    FROM retail
    WHERE CustomerID IS NOT NULL
      AND Quantity > 0
      AND UnitPrice > 0
    GROUP BY CustomerID, DATE(InvoiceDate)
)
SELECT
    ROUND(AVG(DATEDIFF('day', prev_date, purchase_date)), 1) AS avg_days_between_purchases,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY DATEDIFF('day', prev_date, purchase_date)
    ), 1) AS median_days,
    MIN(DATEDIFF('day', prev_date, purchase_date)) AS min_days,
    MAX(DATEDIFF('day', prev_date, purchase_date)) AS max_days
FROM order_dates
WHERE prev_date IS NOT NULL
"""

con.execute(query).fetchdf()

,avg_days_between_purchases,median_days,min_days,max_days
0,45.7,28.0,1,366


The gap distribution is right-skewed: **median = 28 days, mean = 45.7 days**.

The gap between median and mean tells a specific story: most customers who return do so within 28 days (they didn't need a nudge), but a long tail of slow returners pulls the mean up to 45.7 days. This tail is the addressable segment — customers who *might* return if prompted, but won't do so on their own within 30 days.

**Implication for coupon design (population level):**
- A 30-day expiry fires before the slow returners have had time to decide → wasted
- A 90-day expiry removes urgency → customers procrastinate and conversion drops
- A **45-day expiry** sits inside the peak return window (day 31–60) and aligns with the mean gap — it creates urgency at exactly the point where the marginal customer is most persuadable

> This is the *population-level* rationale. Whether the optimal window differs by segment is tested in Case 4 and validated statistically in `03_ab_test_simulation.ipynb`.


### Purchase frequency distribution

In [9]:
query = """
WITH customer_orders AS (
    SELECT 
        CustomerID,
        COUNT(DISTINCT DATE(InvoiceDate)) AS active_days
    FROM retail
    GROUP BY CustomerID
)

SELECT 
    CASE 
        WHEN active_days = 1 THEN '1 purchase'
        WHEN active_days = 2 THEN '2 purchases'
        WHEN active_days <= 5 THEN '3-5 purchases'
        ELSE '6+ purchases'
    END AS purchase_bucket,
    COUNT(*) AS customers
FROM customer_orders
GROUP BY purchase_bucket
ORDER BY purchase_bucket
"""
con.execute(query).fetchdf()

,purchase_bucket,customers
0,1 purchase,1548
1,2 purchases,874
2,3-5 purchases,1117
3,6+ purchases,799


1,548 customers (35.7%) bought exactly once and never returned. This is the primary target population for any win-back campaign. The 874 two-time buyers and 1,117 three-to-five-time buyers represent customers who are already in the habit of returning — intervention here has lower marginal value.


---

## Case 3: Purchase Funnel — Where Does the Business Lose Customers?

### Business question
At each successive purchase, what share of customers continue? Is the drop-off concentrated at one step, or distributed evenly across the funnel?

This is a structural question. If churn is concentrated at one step, that step is the highest-leverage intervention point. If it is distributed evenly, the problem is systemic and harder to address with a single tactic.


In [16]:
query = """
WITH order_sequence AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        InvoiceNo,
        DATE(InvoiceDate) AS purchase_date,
        ROW_NUMBER() OVER (
            PARTITION BY CAST(CustomerID AS INTEGER) 
            ORDER BY DATE(InvoiceDate)
        ) AS purchase_rank
    FROM retail
    WHERE CustomerID IS NOT NULL 
      AND Quantity > 0
    GROUP BY CustomerID, InvoiceNo, DATE(InvoiceDate)
)
SELECT
    COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END) AS made_1st,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) AS made_2nd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) AS made_3rd,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) AS made_4th,
    COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) AS made_5th,

    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 1 THEN CustomerID END), 1) AS "1→2_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 2 THEN CustomerID END), 1) AS "2→3_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 3 THEN CustomerID END), 1) AS "3→4_%",
    ROUND(COUNT(DISTINCT CASE WHEN purchase_rank >= 5 THEN CustomerID END) * 100.0 /
          COUNT(DISTINCT CASE WHEN purchase_rank >= 4 THEN CustomerID END), 1) AS "4→5_%"
FROM order_sequence
"""

con.execute(query).fetchdf()

,made_1st,made_2nd,made_3rd,made_4th,made_5th,1→2_%,2→3_%,3→4_%,4→5_%
0,4338,2845,2010,1502,1114,65.6,70.7,74.7,74.2


| Step | Customers remaining | Step conversion | Drop-off |
|------|--------------------|-----------------| --------|
| 1st purchase | 4,338 | — | — |
| 2nd purchase | 2,845 | 65.6% | **34.4% lost** ← steepest drop |
| 3rd purchase | 2,010 | 70.7% | 29.3% lost |
| 4th purchase | 1,502 | 74.7% | 25.3% lost |
| 5th purchase | 1,114 | 74.2% | 25.8% lost |

**The 1→2 transition loses 34.4% of customers — nearly 10 percentage points more than any subsequent step.** From the 3rd purchase onward, conversion stabilises in the 70–75% range.

This is a classic new-customer activation problem. The business is good at retaining customers who have already made two purchases — the failure point is converting the first purchase into a habit. Every tactic in Cases 1 and 2 ultimately points here: the 45-day coupon is only valuable insofar as it closes this specific gap.


In [2]:
#export csv for tableau
import pandas as pd

data = {
    'Purchase': ['1st', '2nd', '3rd', '4th', '5th'],
    'Customers': [4338, 2845, 2010, 1502, 1114],
    'Offset': [-(4338/2), -(2845/2), -(2010/2), -(1502/2), -(1114/2)]
}
df = pd.DataFrame(data)
df.to_csv('funnel_data.csv', index=False)
print(df)

  Purchase  Customers  Offset
0      1st       4338 -2169.0
1      2nd       2845 -1422.5
2      3rd       2010 -1005.0
3      4th       1502  -751.0
4      5th       1114  -557.0


---

## Case 4: Behavioural Analysis by Segment — What Should We Actually Do?

### Business question
Do the four RFM segments differ meaningfully in their purchase behaviour — and if so, does the population-level 45-day coupon recommendation still hold, or does each segment need a different approach?

Cases 1–3 treated all 4,338 customers as a single population. That is useful for identifying the problem but not for designing the solution. A Whale customer (avg 82 orders, recency 7 days) and an At-risk customer (avg 1.5 orders, recency 248 days) are not the same problem — and should not receive the same intervention.


### Load cluster labels

In [15]:
# Load RFM cluster labels from the segmentation notebook
# Register in DuckDB
con.execute("CREATE TABLE rfm_clusters AS SELECT * FROM read_csv_auto('rfm_clusters.csv')")

# Verify
con.execute("SELECT * FROM rfm_clusters LIMIT 5").fetchdf()

,CustomerID,Cluster
0,12346.0,3
1,12347.0,0
2,12348.0,0
3,12349.0,0
4,12350.0,1


### Full RFM + behavioural profile by segment

In [ ]:
query = """
WITH retail_stats AS (
    SELECT
        CAST(CustomerID AS INTEGER) AS CustomerID,
        COUNT(DISTINCT InvoiceNo)                                          AS num_orders,
        ROUND(SUM(Quantity * UnitPrice), 2)                                AS total_revenue,
        ROUND(SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo), 2)    AS avg_order_value,
        COUNT(DISTINCT StockCode)                                          AS unique_products,
        COUNT(DISTINCT DATE(InvoiceDate))                                  AS active_days,
        MIN(DATE(InvoiceDate))                                             AS first_purchase,
        MAX(DATE(InvoiceDate))                                             AS last_purchase,
        DATEDIFF('day', MIN(DATE(InvoiceDate)), MAX(DATE(InvoiceDate)))    AS customer_lifespan_days
    FROM retail
    WHERE Quantity > 0 AND UnitPrice > 0 AND CustomerID IS NOT NULL
    GROUP BY CustomerID
),
clusters AS (
    SELECT CAST(CAST(CustomerID AS FLOAT) AS INTEGER) AS CustomerID, Cluster
    FROM rfm_clusters
),
combined AS (
    SELECT
        CASE c.Cluster
            WHEN 2 THEN 'C2: Whale'
            WHEN 3 THEN 'C3: High-value loyal'
            WHEN 0 THEN 'C0: Regular'
            WHEN 1 THEN 'C1: At-risk / lapsed'
        END AS segment,
        c.Cluster,
        r.*
    FROM retail_stats r
    JOIN clusters c ON r.CustomerID = c.CustomerID
)
SELECT
    segment,
    COUNT(*)                                        AS customers,
    ROUND(AVG(total_revenue), 0)                    AS avg_revenue,
    ROUND(AVG(avg_order_value), 0)                  AS avg_aov,
    ROUND(AVG(num_orders), 1)                       AS avg_orders,
    ROUND(AVG(unique_products), 0)                  AS avg_unique_products,
    ROUND(AVG(active_days), 1)                      AS avg_active_days,
    ROUND(AVG(customer_lifespan_days), 0)           AS avg_lifespan_days,
    ROUND(AVG(CASE WHEN num_orders > 1
        THEN customer_lifespan_days * 1.0 / (num_orders - 1)
        ELSE NULL END), 0)                          AS avg_gap_between_orders
FROM combined
GROUP BY segment, Cluster
ORDER BY avg_revenue DESC
"""

con.execute(query).fetchdf()


Reading this table across all columns reveals the distinct behavioural signature of each segment:

| Segment | Customers | Avg Revenue | Avg AOV | Avg Orders | Avg Gap (days) | Avg Recency (days) | Interpretation |
|---------|-----------|-------------|---------|------------|----------------|--------------------|----------------|
| C2: Whale | 13 | £127,338 | £8,571 | 82.5 | ~7 | 7 | Bulk / B2B buyers. Order weekly. Never really lapse — recency of 7 days means they are almost certainly active. Any churn event would be immediately visible and catastrophic. |
| C3: High-value loyal | 204 | £12,709 | £1,080 | 22.3 | ~16 | 15.5 | High-frequency, high-AOV retail buyers. Gap of ~16 days means they are in an established purchase habit. Recency of 15.5 days confirms they are still active. These customers do not need a win-back coupon — they need a reason to stay. |
| C0: Regular | 3,054 | £1,359 | £377 | 3.7 | ~43 | 43.7 | The largest group. Moderate frequency, gap closely matches recency — meaning they are roughly on schedule. A 30-day coupon issued after first purchase would arrive just before their natural return point, creating a nudge without being redundant. |
| C1: At-risk / lapsed | 1,067 | £481 | £315 | 1.6 | ~48 | 248 | Historical gap of ~48 days but recency of 248 days — they have been gone for 5× their normal cycle. They have not just missed a purchase; they have structurally lapsed. A short 30-day coupon would expire before it can do any work. They need a longer window (45 days) that aligns with their historical purchase rhythm and creates urgency at the right moment. |

**The key distinction between C0 and C1 is not AOV or order frequency — it is the relationship between historical gap and current recency.** C0 customers are roughly on schedule; C1 customers are far beyond their normal cycle and need a fundamentally different approach.


### Segment-specific purchase gap analysis

The avg_gap_between_orders above is approximate (lifespan / orders). A more precise view uses LAG() to calculate actual inter-purchase gaps per cluster.


In [ ]:
query = """
WITH order_dates AS (
    SELECT
        CAST(r.CustomerID AS INTEGER) AS CustomerID,
        DATE(r.InvoiceDate) AS purchase_date,
        LAG(DATE(r.InvoiceDate)) OVER (
            PARTITION BY CAST(r.CustomerID AS INTEGER)
            ORDER BY DATE(r.InvoiceDate)
        ) AS prev_date,
        CASE c.Cluster
            WHEN 2 THEN 'C2: Whale'
            WHEN 3 THEN 'C3: High-value loyal'
            WHEN 0 THEN 'C0: Regular'
            WHEN 1 THEN 'C1: At-risk / lapsed'
        END AS segment
    FROM retail r
    JOIN (
        SELECT CAST(CAST(CustomerID AS FLOAT) AS INTEGER) AS CustomerID, Cluster
        FROM rfm_clusters
    ) c ON CAST(r.CustomerID AS INTEGER) = c.CustomerID
    WHERE r.CustomerID IS NOT NULL AND r.Quantity > 0 AND r.UnitPrice > 0
    GROUP BY r.CustomerID, DATE(r.InvoiceDate), c.Cluster
),
gaps AS (
    SELECT
        segment,
        DATEDIFF('day', prev_date, purchase_date) AS gap_days
    FROM order_dates
    WHERE prev_date IS NOT NULL
)
SELECT
    segment,
    COUNT(*)                                                                AS gap_observations,
    ROUND(AVG(gap_days), 1)                                                 AS mean_gap,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY gap_days), 1)        AS median_gap,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY gap_days), 1)       AS p25_gap,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY gap_days), 1)       AS p75_gap
FROM gaps
GROUP BY segment
ORDER BY median_gap
"""

con.execute(query).fetchdf()


This per-cluster gap distribution is the direct data input for coupon window design:

- **C2 Whale** and **C3 Loyal**: short median gaps (< 20 days). These segments are already in a purchase habit. Coupons are largely wasted here — the marginal effect is low because they would have returned anyway.
- **C0 Regular**: median gap ~43 days. A 30-day coupon creates urgency just before their natural return point — the intervention arrives when the customer is already considering a return trip.
- **C1 At-risk**: median gap ~48 days *when they were active*, but current recency of 248 days. The historical gap tells us what their purchase rhythm *was* — a 45-day coupon aligns with that rhythm and gives them a concrete deadline to re-engage before the window closes.

> **This analysis establishes the rationale for segment-specific coupon windows.** The statistical validation — whether the coupon lift is real and not due to chance — is tested in `03_ab_test_simulation.ipynb` using simulated A/B experiments with t-tests and 95% confidence intervals.


---
---

## Summary

Four questions, one thread: **where does this business lose customers, and what is the most precise intervention for each type?**

---

**Case 1 — How many customers ever return?**

64.3% make at least one repeat purchase. Healthy — but a lifetime metric. It tells us retention *exists*, not whether the business can influence *when* it happens. The real question is timing.

---

**Case 2 — When is the right moment to intervene?**

The 31–60 day window captures +14.9pp in incremental returns — nearly double the 61–90 day window (+8.6pp). The inter-purchase gap is right-skewed: median = 28 days, mean = 45.7 days. Most customers who return do so quickly and without a nudge. The addressable segment is the slow-return tail — customers who *might* return if prompted, but won't do so on their own within 30 days.

A 30-day coupon expires before this window opens. A 90-day coupon removes urgency. A **45-day window** sits inside the peak return zone while forcing a decision before the moment passes.

---

**Case 3 — Where in the funnel is the biggest loss?**

| Step | Customers | Drop-off |
|------|-----------|---------|
| 1st → 2nd purchase | 4,338 → 2,845 | **−34.4%** ← steepest |
| 2nd → 3rd | 2,845 → 2,010 | −29.3% |
| 3rd → 4th | 2,010 → 1,502 | −25.3% |
| 4th → 5th | 1,502 → 1,114 | −25.8% |

The 1→2 transition is nearly 10pp worse than any subsequent step. From the 3rd purchase onward, retention stabilises at ~70–75%. **The problem is activation, not long-term loyalty.** Every intervention should target the first purchase — not customers who already have a buying habit.

---

**Case 4 — Does the same 45-day coupon work for everyone?**

No. The critical variable is not AOV or order frequency — it is the ratio of **historical gap to current recency**. A customer whose recency matches their historical gap is on schedule. A customer whose recency is 5× their historical gap has structurally lapsed and needs a fundamentally different approach.

| Segment | Hist. gap | Recency | Ratio | Recommended tactic |
|---------|-----------|---------|-------|--------------------|
| C2: Whale (n=13) | ~7d | 7d | 1× | Dedicated account management. Still active — this is a relationship risk, not an activation problem. Any churn is immediately visible and costly. |
| C3: High-value loyal (n=204) | ~16d | 15.5d | 1× | VIP retention programme. Active and high-value — focus on preventing churn before it happens, not recovering it after. |
| C0: Regular (n=3,054) | ~43d | 43.7d | 1× | **30-day post-first-purchase coupon.** On schedule; a short nudge just before the natural return point is enough. |
| C1: At-risk / lapsed (n=1,067) | ~48d | 248d | **5×** | **45-day win-back coupon.** Structurally lapsed. The 45-day window aligns with their historical purchase rhythm and creates a deadline before the opportunity closes entirely. Prioritise the 60–90 day lapsed sub-segment first — lowest reactivation cost, highest probability of response. |

---

### Limitations

1. **No campaign cost data.** Revenue lift estimates assume zero coupon cost. A 10–15% discount reduces margin; actual ROI depends on discount depth and redemption rate — neither available in this dataset.

2. **Single observation window.** The dataset covers exactly one year (Dec 2010 – Dec 2011). Customers labelled "at-risk" may be seasonal buyers whose next purchase falls outside the window. Longer data would separate structural churn from seasonal absence.

3. **Simulation, not experiment.** Coupon lift figures in `03_ab_test_simulation.ipynb` are derived from historical gap distributions, not a real campaign. Statistical significance holds given the assumptions, but the assumptions themselves — control rate, lift magnitude — are estimates, not observations.

4. **All purchases treated equally.** High-AOV purchases may have different retention dynamics than low-AOV ones. Category-level or product-level variation is not captured here.

---

### What comes next

`03_ab_test_simulation.ipynb` takes the segment-specific coupon windows from Case 4 and runs simulated A/B tests — C0 (30-day) and C1 (45-day) separately — using independent t-tests and 95% confidence intervals. Both results are statistically significant (p < 0.0001 and p = 0.0005 respectively), with an estimated combined revenue lift of **£286,929**.
